Generate a random list of nodes and edges, specifying the strengths distributions

In [1]:
# auto-reload the packages at every run
%load_ext autoreload
%autoreload 2

#display all the results not only the last one
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
N = 300000 #int(2.9 * 1e5)
E = int(2.3 * 1e6)

In [3]:
import numpy as np
import numpy.random as npr
# sample the strengths in from the log-normal distribution
mu_in, scale_in = 9.145050968796784, 3.0501786627011396
npr.seed(0)
logn_sample = lambda mu, scale: npr.lognormal(mean=mu, sigma=scale, size=N)
prop_in = logn_sample(mu_in, scale_in)

# sample the strengths out from the log-normal distribution
mu_out, scale_out = 9.753643239750687, 3.2045983690423174
stre_out_method = 'correlated'
if stre_out_method == 'uncorrelated':
    prop_out = logn_sample(mu_out, scale_out)
elif stre_out_method == 'correlated':
    mu_eps, scale_eps = mu_out - mu_in, np.sqrt(scale_out**2 - scale_in**2)
    eps = logn_sample(mu_eps, scale_eps)
    prop_out = prop_in * eps

In [4]:
# # plot the distributions of the prop_in and prop_out
# import matplotlib.pyplot as plt
# plt.figure(figsize=(10, 5))
# plt.hist(np.log(prop_in), bins=100, alpha=0.5, label='prop_in', density=True)
# plt.hist(np.log(prop_out), bins=100, alpha=0.5, label='prop_out', density=True)
# plt.xlabel('Strength')
# plt.ylabel('Density')
# plt.legend()
# plt.title('Distribution of Strengths')

In [4]:
from graph_ensembles import sparse as sp
import graph_ensembles.utils as utils
# import graph_ensembles.dependencies as dep

In [ ]:
#v = np.arange(len(prop_out), dtype=np.int64)
param = 5e-16 #1.8996372e-17
kwargs_model = {
                "num_vertices" : N,
                "prop_out" : prop_out,
                "prop_in" : prop_in,
                "param" : param,
                "selfloops" : False,
                "name" : 'Invariant',
                "level" : 0,
                "seed" : 0,
                "perc_intra_nodes" : 1,
                "fit_method" : 'num_edges_intra',
                "corpkey" : False,
                "test_graph" : "intra",
                } #git hub repositories
model = sp.ScaleInvariantModel(**kwargs_model)

Parallelized Code

In [12]:
gs = model.sample()

In [17]:
Ns = np.unique(np.concatenate(gs.adj.nonzero())).size

gs.num_edges() / (Ns * (Ns - 1))
gs.num_edges()

np.float64(0.00042903316311098126)

np.int64(17606510)

In [20]:
import os
dataset_folder = f"{os.path.expanduser('~')}/Documents/Datasets/ING-Directed/xgrid_20240427_xtrans_20240424"
full_path = f"{dataset_folder}/rows_cols.npz"
os.makedirs(dataset_folder, exist_ok = True)

# np.savez_compressed(full_path, a=gs.adj.nonzero())
rows, cols = np.load(full_path)["a"]

Without parallelization copied from the ensembles

In [15]:
from numba import njit, prange
from numba.typed import List
@njit()  # pragma: no cover
def _binary_sample(p_ij, param, prop_out, prop_in, prop_dyad, selfloops):
    """Sample from the ensemble."""
    rows = List()
    cols = List()

    N = len(prop_out)
    np.random.seed(1)
    for i in prange(N):
        p_out_i = prop_out[i]
        for j in range(N):
            if (i != j):
                p_in_j = prop_in[j]
                p = p_ij(param, p_out_i, p_in_j, prop_dyad(i, j))
                if np.random.random() < p:
                    rows.append(i)
                    cols.append(j)

        # if i % 10 == 0: print(f'-i: {i}',)
        
    return rows, cols

@njit()  # pragma: no cover
def leo_binary_sample(p_ij, param, prop_out, prop_in, prop_dyad, selfloops):
    """Sample from the ensemble."""
    rows = []
    cols = []
    np.random.seed(1)
    for i, p_out_i in enumerate(prop_out):
        for j, p_in_j in enumerate(prop_in):
            if (i != j) | selfloops:
                p = p_ij(param, p_out_i, p_in_j, prop_dyad(i, j))
                if np.random.random() < p:
                    rows.append(i)
                    cols.append(j)

    return rows, cols

In [ ]:
rows_fail, cols_fail = leo_binary_sample(
    model.p_ij,
    model.param,
    model.prop_out,
    model.prop_in,
    model.prop_dyad,
    model.selfloops,
)
# ipython_pygments_lexers

In [ ]:
# # sort rows, cols first wrt rows and then to cols
# idx = np.lexsort((cols_fail, rows_fail))
# rows_fail = np.array(rows_fail)[idx]
# cols_fail = np.array(cols_fail)[idx]

# idx = np.lexsort((cols, rows))
# rows = rows[idx]
# cols = cols[idx]

# if np.all(rows == rows_fail) and np.all(cols == cols_fail):
#     print(f'-rows and cols: {True}',)
# else:
#     if np.all(rows == rows_fail):
#         print(f'-rows: {True}',)
#     else:
#         rows == rows_fail

#     if np.all(cols == cols_fail):
#         print(f'-cols: {True}',)
#     else:
#         cols == cols_fail

#     rows_fail
#     rows

#     cols_fail
#     cols

-rows and cols: True


Create a Dataset with naics codes similarly to ING one

In [21]:
# full naics codes
low, high = 111150, 814110
naics_codes = np.arange(low, high + 1)

# filter out the 52, 55, 72, 99 as done in the real net
permitted_naics = np.array(naics_codes // int(1e4), dtype = int)
mask = np.isin(permitted_naics, [52, 55, 72, 99])

# create a mask
naics_codes = naics_codes[~mask]

In [22]:
# set the total number of sectors
num_naics = 972
diff_naics = high - low

assert N > num_naics, "The number of sectors (num_naics) should be less than the number of nodes (N)"

# select num_naics from filtered naics_codes, then select N of them (with replacement) to create groups
np.random.seed(0)
num_naics_codes = np.random.choice(a = naics_codes, size = num_naics, replace = False)
id_naics = np.random.choice(a = num_naics_codes, size = N, replace = True)

In [23]:
import pandas as pd

# initialize the pandas DF with rows and cols
rows, cols = gs.adj.nonzero()
pdtrans = pd.DataFrame({"payer_grid_id" : rows, "beneficiary_grid_id" : cols})

def create_naics_col(df, pay_ben, id_naics):
    df[f"{pay_ben}_naics_code"] = df[f"{pay_ben}_grid_id"].map(lambda i: id_naics[i])
    df[f"{pay_ben}_naics_desc"] = pay_ben[0]
    return df

# create payer / beneficiaries naics columns, nrofpayments
pdtrans = create_naics_col(pdtrans, "payer", id_naics)
pdtrans = create_naics_col(pdtrans, "beneficiary", id_naics)
pdtrans.loc[:, "nrofpayments"] = 1

# assign the weights with the MaxEnt rule
W = np.sum(prop_out)
pdtrans.loc[:, "amount_euro"] = pdtrans.apply(lambda row: prop_out[row["payer_grid_id"]] * prop_in[row["beneficiary_grid_id"]], axis=1)
pdtrans.loc[:, "amount_euro"] /= W

# reorder the pd.DataFrame
pdtrans = pdtrans.loc[:, ["payer_grid_id", "payer_naics_code", "payer_naics_desc", "beneficiary_grid_id", "beneficiary_naics_code", "beneficiary_naics_desc", "nrofpayments", "amount_euro"]]

In [24]:
import os
dataset_folder = f"{os.path.expanduser('~')}/Documents/Datasets/ING-Directed/xgrid_20240427_xtrans_20240424"
full_path = f"{dataset_folder}/pdtrans_no_rotw_gridSelfLoops_52559299_grid_id.csv"
os.makedirs(dataset_folder, exist_ok = True)

if True: #not os.path.exists(full_path):
    pdtrans.to_csv(full_path, index = False)

In [ ]:
density = lambda N, E: E / (N * (N-1))

N, E, density(N, E)

Ns = utils.nodes_from(pdtrans, id_code="grid_id").size
Es = pdtrans.shape[0]
Ns, Es, density(Ns, Es)

(202578, 17606510, 0.00042903316311098126)

(300000, 2300000, 2.5555640741024693e-05)

Old Code

In [ ]:
import numpy as np
from concurrent.futures import ThreadPoolExecutor

def worker(start, end, N, p_ij, param, prop_out, prop_in, prop_dyad, selfloops):
    rows = []
    cols = []
    worker_id = current_thread().name
    print(f'Worker {worker_id} started: processing indices {start} to {end}')
    for flat_idx in range(start, end):
        # Convert flat index to 2D indices, via CSR indexing
        i = flat_idx // N
        j = flat_idx % N

        # if selfloops is False and i == j --> skip this pair
        if not selfloops and i == j:
            continue
        p = p_ij(param, prop_out[i], prop_in[j], prop_dyad(i, j))
        if np.random.random() < p:
            rows.append(i)
            cols.append(j)
        # Optionally, print progress for each worker
        # print(f'Worker {worker_id}, rows, cols: {rows}, {cols}')
    print(f'Worker {worker_id} finished.')
    return np.array(rows, dtype=np.int64), np.array(cols, dtype=np.int64)

def parallel_sample(p_ij, param, prop_out, prop_in, prop_dyad, selfloops, num_threads=3):
    N = len(prop_out)
    total_ops = N * N  # or N * (N - 1) if not selfloops
    num_chunks = total_ops // num_threads

    futures = []
    with ThreadPoolExecutor(max_workers=num_threads, thread_name_prefix='agent') as executor:
        for t in range(num_threads):
            start = t * num_chunks
            end = (t + 1) * num_chunks if t < num_threads - 1 else total_ops
            futures.append(executor.submit(
                worker, start, end, N, p_ij, param, prop_out, prop_in, prop_dyad, selfloops
            ))

    rows = []
    cols = []
    for f in futures:
        r, c = f.result()
        rows.append(r)
        cols.append(c)
    return np.concatenate(rows), np.concatenate(cols)

In [ ]:
import numpy as np
import numba
from numba import njit, prange, config

@njit(parallel=True)
def _fixed_binary_sample(p_ij, param, prop_out, prop_in, prop_dyad, selfloops):
    N = len(prop_out)
    
    # Pre-allocate thread-local buffers as arrays
    n_threads = config.NUMBA_DEFAULT_NUM_THREADS
    max_edges_per_thread = 2000  # Adjust based on expected density
    thread_buffers_rows = np.empty((n_threads, max_edges_per_thread), dtype=np.int64)
    thread_buffers_cols = np.empty((n_threads, max_edges_per_thread), dtype=np.int64)
    
    # counts how many edges were sampled by each thread
    counts = np.zeros(n_threads, dtype=np.int64)

    np.random.seed(0)  # Set seed for reproducibility when parallel = False
    # Parallel sampling
    for i in prange(N):
        thread_id = numba.get_thread_id()
        # print(f'-thread_id: {thread_id}',)
        p_out_i = prop_out[i]
        
        for j in range(N):
            if i != j:
                p_in_j = prop_in[j]
                p = p_ij(param, p_out_i, p_in_j, prop_dyad(i, j))
                if np.random.random() < p:
                    if counts[thread_id] < max_edges_per_thread:
                        thread_buffers_rows[thread_id, counts[thread_id]] = i
                        thread_buffers_cols[thread_id, counts[thread_id]] = j
                        counts[thread_id] += 1

    # Concatenate results
    total_edges = np.sum(counts)
    rows = np.empty(total_edges, dtype=np.int64)
    cols = np.empty(total_edges, dtype=np.int64)
    
    idx = 0
    for t in range(n_threads):
        n = counts[t]
        rows[idx:idx+n] = thread_buffers_rows[t, :n]
        cols[idx:idx+n] = thread_buffers_cols[t, :n]
        idx += n

    return rows, cols


In [ ]:
from numba import njit, prange

@njit(parallel=True)
def _fast_binary_sample(p_ij, param, prop_out, prop_in, prop_dyad, selfloops, randmat):
    N = len(prop_out)
    edge_counts = np.zeros(N, dtype=np.int64)
    
    # First pass: count edges per i
    for i in prange(N):
        for j in range(N):
            if i != j:
                p = p_ij(param, prop_out[i], prop_in[j], prop_dyad(i, j))
                if randmat[i,j] < p:
                    edge_counts[i] += 1
    total_edges = np.sum(edge_counts)
    
    # Compute prefix sum for unique indexing
    idxs = np.zeros(N, dtype=np.int64)
    idxs[1:] = np.cumsum(edge_counts)[:-1]
    rows = np.empty(total_edges, dtype=np.int64)
    cols = np.empty(total_edges, dtype=np.int64)
    
    # Second pass: fill arrays
    for i in prange(N):
        idx = idxs[i]
        for j in range(N):
            if i != j:
                p = p_ij(param, prop_out[i], prop_in[j], prop_dyad(i, j))
                if randmat[i,j] < p:
                    rows[idx] = i
                    cols[idx] = j
                    idx += 1
    return rows, cols

In [ ]:
rng = np.random.default_rng(0)
randmat = rng.random((N, N))

rows_fast, cols_fast = _fast_binary_sample(
    self.p_ij,
    self.param,
    self.prop_out,
    self.prop_in,
    self.prop_dyad,
    self.selfloops,
    randmat
)
# ipython_pygments_lexers